# Logistic Regression — Implementations

The binary and multi-class classifiers again. The torch lanes hand the calculus to `torch.func` and `F.cross_entropy` and keep everything else the notebook chose; the library lane switches sklearn's default penalty *off*, because comparing a regularised fit with an MLE is comparing two different estimators. sklearn's multinomial fit is exactly that situation — a different estimator from 500 fixed GD steps — so softmax deliberately has no library lane rather than a lane that fails its own comparison.

## 04_logistic_regression

Binary classification by Newton.

### torch

**What torch adds:** `torch.func.grad` and `torch.func.hessian` replace the two hand-derived formulas, and the score-equation check shows the optimum is the same one.

In [ ]:
import numpy as np
import torch
from torch.func import grad, hessian

# hints:
# 1. binary_cross_entropy takes logits, not probabilities — keep it that way.
# 2. logaddexp(0, z) is log(1+e^z) with no overflow; same trick as log1pexp.
# 3. torch.func.grad and torch.func.hessian differentiate the loss for you.
# 4. Newton solves H·δ = g; keep the tiny damping term the notebook uses.
# 5. At the optimum the path does not matter — only that both lanes converge.


def sigmoid(z):
    return torch.sigmoid(z)


def binary_cross_entropy(z, y):
    """Mean BCE on logits, the overflow-safe way: log(1+e^z) − y·z."""
    return torch.mean(torch.logaddexp(torch.zeros_like(z), z) - y * z)


class LogisticRegressionScratch:
    """Newton's method with the gradient and Hessian coming from torch.func
    instead of the hand-derived formulas — the entire difference between this
    lane and the NumPy one is who does the calculus."""

    def __init__(self, solver="newton", lam=0.0, n_iter=200, tol=1e-8, eta=None):
        self.solver = solver
        self.lam = lam
        self.n_iter = n_iter
        self.tol = tol

    @staticmethod
    def _add_intercept(X):
        return torch.cat([torch.ones(X.shape[0], 1, dtype=X.dtype), X], dim=1)

    def fit(self, X, y):
        Xi = self._add_intercept(torch.as_tensor(np.asarray(X, dtype=float)))
        yt = torch.as_tensor(np.asarray(y, dtype=float))
        n, p = Xi.shape

        def loss(theta):
            return binary_cross_entropy(Xi @ theta, yt)

        theta = torch.zeros(p, dtype=torch.float64)
        self.loss_history_ = []
        for _ in range(self.n_iter):
            g = grad(loss)(theta)
            H = hessian(loss)(theta)
            delta = torch.linalg.solve(H + 1e-10 * torch.eye(p, dtype=H.dtype), g)
            theta = theta - delta
            self.loss_history_.append(float(loss(theta)))
            if float(torch.linalg.norm(g)) < self.tol:
                break

        self.theta_ = theta.numpy()
        self.intercept_ = float(theta[0])
        self.coef_ = theta[1:].numpy()
        return self

    def predict_proba(self, X):
        Xi = self._add_intercept(torch.as_tensor(np.asarray(X, dtype=float)))
        return sigmoid(Xi @ torch.as_tensor(self.theta_)).numpy()

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


In [ ]:
# exports: coef, intercept
_rng_eq = np.random.default_rng(3)
X_eq = _rng_eq.normal(size=(200, 4))
_p_eq = 1.0 / (1.0 + np.exp(-(X_eq @ np.array([1.0, -1.5, 0.5, 2.0]) - 0.3)))
y_eq = (_rng_eq.uniform(size=200) < _p_eq).astype(float)

_fit_eq = LogisticRegressionScratch(solver="newton", n_iter=50).fit(X_eq, y_eq)
coef, intercept = _fit_eq.coef_, _fit_eq.intercept_
print("accuracy:", float((_fit_eq.predict(X_eq) == y_eq).mean()))


In [ ]:
assert (_fit_eq.predict(X_eq) == y_eq).mean() > 0.85, "separably generated data, high accuracy"
assert np.all(np.diff(_fit_eq.loss_history_[:5]) < 0), "Newton drops the loss fast and early"

# The MLE's signature: at the optimum, Xᵀ(p − y) = 0, including the intercept column.
_p_hat = _fit_eq.predict_proba(X_eq)
_Xi = np.hstack([np.ones((200, 1)), X_eq])
assert np.max(np.abs(_Xi.T @ (_p_hat - y_eq))) < 1e-6, "score equations hold at the optimum"


### library

sklearn with `penalty=None` — the default C=1.0 is a silently different estimator, and turning it off is the lane's one lesson.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# hints:
# 1. sklearn regularises BY DEFAULT (C=1.0); penalty=None asks for the plain MLE.
# 2. Newton and lbfgs find the same optimum — the loss is convex, tol decides how close.
# 3. C is an inverse penalty: bigger C, less shrinkage — opposite direction to λ.


class LogisticRegressionScratch:
    """sklearn with the default penalty switched off, because the notebook's
    estimator is the unregularised MLE and the comparison is only fair if both
    lanes optimise the same objective."""

    def __init__(self, solver="newton", lam=0.0, n_iter=200, tol=1e-8, eta=None):
        self.n_iter = n_iter
        self.tol = tol

    def fit(self, X, y):
        self._model = LogisticRegression(penalty=None, tol=1e-10, max_iter=5000)
        self._model.fit(np.asarray(X), np.asarray(y).astype(int))
        self.coef_ = self._model.coef_[0]
        self.intercept_ = float(self._model.intercept_[0])
        return self

    def predict_proba(self, X):
        return self._model.predict_proba(np.asarray(X))[:, 1]

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


In [ ]:
# exports: coef, intercept
_rng_eq = np.random.default_rng(3)
X_eq = _rng_eq.normal(size=(200, 4))
_p_eq = 1.0 / (1.0 + np.exp(-(X_eq @ np.array([1.0, -1.5, 0.5, 2.0]) - 0.3)))
y_eq = (_rng_eq.uniform(size=200) < _p_eq).astype(float)

_fit_eq = LogisticRegressionScratch().fit(X_eq, y_eq)
coef, intercept = _fit_eq.coef_, _fit_eq.intercept_


In [ ]:
_p_hat = _fit_eq.predict_proba(X_eq)
_Xi = np.hstack([np.ones((200, 1)), X_eq])
assert np.max(np.abs(_Xi.T @ (_p_hat - y_eq))) < 1e-4, "same score equations, so same optimum"


## 04_softmax_regression

The K-class generalisation.

### torch

`F.cross_entropy` fuses the softmax and the log-loss (log-sum-exp inside, so no overflow) and autograd returns the (P − Y)ᵀX/n the notebook derives.

In [ ]:
import numpy as np
import torch

# hints:
# 1. cross_entropy takes raw logits and integer labels — no one-hot, no softmax.
# 2. Zeros init and eta·g steps, to stay comparable with the NumPy lane's path.
# 3. The gradient of mean cross-entropy is exactly (P − Y)ᵀX/n — what the notebook derived.
# 4. reduction='mean' matters: 'sum' scales the gradient by n and the lr is wrong.


def softmax(Z):
    return torch.softmax(Z, dim=-1)


class SoftmaxRegressionScratch:
    """The same fixed-step gradient descent, with `F.cross_entropy` producing
    the gradient the notebook derives by hand ((P − Y)ᵀX / n)."""

    def __init__(self, lam=0.0, eta=0.5, n_iter=4000, tol=1e-7):
        self.lam = lam
        self.eta = eta
        self.n_iter = n_iter
        self.tol = tol

    @staticmethod
    def _add_intercept(X):
        return torch.cat([torch.ones(X.shape[0], 1, dtype=X.dtype), X], dim=1)

    def fit(self, X, y):
        Xi = self._add_intercept(torch.as_tensor(np.asarray(X, dtype=float)))
        yt = torch.as_tensor(np.searchsorted(np.unique(y), np.asarray(y)))
        self.classes_ = np.unique(y)
        n, p = Xi.shape
        K = len(self.classes_)

        Theta = torch.zeros((K, p), dtype=torch.float64, requires_grad=True)
        self.loss_history_ = []
        for _ in range(self.n_iter):
            loss = torch.nn.functional.cross_entropy(Xi @ Theta.T, yt)
            loss.backward()
            with torch.no_grad():
                g_norm = float(torch.linalg.norm(Theta.grad))
                Theta -= self.eta * Theta.grad
                Theta.grad = None
            self.loss_history_.append(float(loss))
            if g_norm < self.tol:
                break

        self.Theta_ = Theta.detach().numpy()
        return self

    def predict_proba(self, X):
        Xi = self._add_intercept(torch.as_tensor(np.asarray(X, dtype=float)))
        return softmax(Xi @ torch.as_tensor(self.Theta_).T).numpy()

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


In [ ]:
# exports: Theta
_rng_eq = np.random.default_rng(9)
X_eq = np.vstack([_rng_eq.normal(loc, 0.6, size=(40, 2)) for loc in (-2.0, 0.0, 2.0)])
y_eq = np.repeat([0, 1, 2], 40)

_fit_eq = SoftmaxRegressionScratch(eta=0.5, n_iter=500, tol=0.0).fit(X_eq, y_eq)
Theta = _fit_eq.Theta_
print("train accuracy:", float((_fit_eq.predict(X_eq) == y_eq).mean()))


In [ ]:
assert (_fit_eq.predict(X_eq) == y_eq).mean() > 0.9, "three well-separated blobs"
_P = _fit_eq.predict_proba(X_eq)
assert np.allclose(_P.sum(axis=1), 1.0, atol=1e-9), "rows of P are distributions"
assert _fit_eq.loss_history_[-1] < _fit_eq.loss_history_[0], "the loss went down"
